# Поглавље 7: Прављење ћаскања апликација
## Microsoft Foundry Models API Брзи почетак

Овај блок белешки је прилагођен из [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) која садржи блок белешке које приступају [Azure OpenAI](notebook-azure-openai.ipynb) услугама.

> **Напомена:** GitHub Models се укида крајем јула 2026. Овај блок белешки сада циља на [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), који нуди исти каталог модела бесплатних за пробу и искуство Azure AI Inference SDK.


# Преглед  
"Велики језички модели су функције које мапирају текст у текст. Дата улазна низ текста, велики језички модел покушава да предвиди текст који ће уследити"(1). Овај "брзи почетак" нотебоок ће корисницима представити основне појмове LLM, кључне захтеве пакета за почетак са AML, благи увод у дизајн упита, и неколико кратких примера различитих случајева употребе. 


## Садржај  

[Преглед](#overview)  
[Како користити OpenAI Service](#how-to-use-openai-service)  
[1. Креирање вашег OpenAI Service](#1.-creating-your-openai-service)  
[2. Инсталација](#2.-installation)    
[3. Акредитиви](#3.-credentials)  

[Примери коришћења](#use-cases)    
[1. Сажми текст](#1.-summarize-text)  
[2. Класификуј текст](#2.-classify-text)  
[3. Генериши нове називе производа](#3.-generate-new-product-names)  
[4. Финитирај класификатор](#4.fine-tune-a-classifier)  

[Референце](#references)


### Направите свој први упит  
Ова кратка вежба ће пружити основни увод за слање упита моделу у Microsoft Foundry Models за једноставан задатак „сумирање“.  


**Кораци**:  
1. Инсталирајте `azure-ai-inference` библиотеку у своје python окружење, ако то још нисте урадили.  
2. Учитајте стандардне помоћне библиотеке и подесите своје Microsoft Foundry Models акредитиве.  
3. Изаберите модел за свој задатак  
4. Направите једноставан упит за модел  
5. Пошаљите свој захтев модел API-ју!  


### 1. Инсталирајте `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Увези помоћне библиотеке и иницијализуј креденцијале


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Проналажење правог модела  
Модели попут GPT-4o и GPT-4o мини могу разумети и генерисати природни језик и доступни су у каталогу Microsoft Foundry Models заједно са моделима из Meta, Mistral, Cohere и Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Дизајн упита  

"Магија великих језичких модела је у томе што, обучавајући се да минимизирају ову грешку предвиђања на огромним количинама текста, модели на крају уче концепте корисне за ова предвиђања. На пример, они уче концепте као што су"(1):

* како се пише
* како функционише граматика
* како да се препричава
* како да се одговара на питања
* како да се води разговор
* како да се пише на многим језицима
* како да се кодира
* итд.

#### Како контролисати велики језички модел  
"Од свих улаза у велики језички модел, далеко највише утичући је текстуални упит(1).

Велики језички модели могу бити упућени да производе излаз на неколико начина:

Упутство: Реците моделу шта желите
Завршетак: Подстакните модел да заврши почетак онога што желите
Демонстрација: Пokaжите моделу шта желите, са или:
Неколико примера у упиту
Много стотина или хиљада примера у скупу тренинг података за фино подешавање"



#### Постоје три основне смернице за креирање упита:

**Покажи и реци**. Јасно наведите шта желите путем упутстава, примера или комбинације оба. Ако желите да модел рангира листу предмета по азбучном реду или да класификује пасус по сентименту, покажите му да је то оно што желите.

**Обезбедите квалитетне податке**. Ако покушавате да направите класификатор или да натерате модел да следи образац, уверите се да има довољно примера. Обавезно прочитајте своје примере — модел је обично довољно паметан да препозна основне правописне грешке и да вам да одговор, али такође може претпоставити да је то намерно и то може утицати на одговор.

**Проверите поставке.** Поставке temperature и top_p контролишу колико је модел детерминистички у генерисању одговора. Ако тражите одговор где постоји само један тачан одговор, онда треба да их подесите ниже. Ако тражите разноврсније одговоре, онда желите да их подесите више. Најчешћа грешка људи са овим подешавањима је размишљање да су то контроле "паметности" или "креативности".


Извор: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Пошаљи!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Поновите исти позив, како се резултати упоређују?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Сажми текст  
#### Изазов  
Сажми текст додавањем 'tl;dr:' на крају текста. Обрати пажњу како модел разуме како да изведе бројне задатке без додатних упутстава. Можеш експериментисати са описнијим упутствима од tl;dr да модификујеш понашање модела и прилагодиш добијени резиме(3).  

Недавни рад је показао значајан напредак у многим NLP задацима и референтним тестовима предобуком на великом корпусу текста, праћеном прецизним подешавањем за одређени задатак. Иако је архитектура обично независна од задатка, овај метод и даље захтева скупове података за прецизно подешавање специфичне за задатак од хиљада или десетина хиљада примера. За разлику од тога, људи генерално могу извршити нови језички задатак из само неколико примера или једноставних упутстава - нешто што тренутним NLP системима углавном још увек представља изазов. Овде показујемо да повећавање величине језичких модела значајно побољшава перформансе без прецизног подешавања за задатке на које нису специјализовани, понекад чак достижући конкурентност са претходним врхунским методама прецизног подешавања. 



Tl;dr


# Вежбе за неколико случајева употребе  
1. Сажми текст  
2. Класификуј текст  
3. Креирај нове називе производа


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Класификујте Текст  
#### Изазов  
Класификујте ставке у категорије које се обезбеђују у време извршавања. У следећем примеру, обезбеђујемо и категорије и текст који треба класификовати у упиту (*playground_reference). 

Упит купца: Здраво, један тастер на тастатури мог лаптопа недавно се покварио и потребна ми је замена:

Класификована категорија:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Генериши нова имена производа
#### Изазов
Креирај имена производа од примера речи. Овде у промпт укључујемо информације о производу за који ћемо генерисати имена. Такође пружамо сличан пример да покажемо образац који желимо да добијемо. Поставили смо и вредност температуре високо како бисмо повећали случајност и добили иновативније одговоре.

Опис производа: Кућни направљач шејкова
Почетне речи: брз, здрав, компактни.
Имена производа: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис производа: Пар ципела који може да стане на било коју величину стопала.
Почетне речи: прилагодљив, прилагођен, свестран.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Референце  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry портал](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Најбоље праксе за фино подешавање GPT модела за класификацију текста](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# За додатну помоћ  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Сарадници
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
